# Challenge 4 – Agent Workflow

## Goal

Build a Google ADK workflow that answers a user question, critiques the initial answer, and refines the response before returning the final result.

## Workflow

User Request  
↓  
Greeter Agent  
↓  
Search Agent  
↓  
Critique Agent  
↓  
Refine Agent  
↓  
Final Response

## 1. Environment Setup

In [1]:
import os

from google.adk.agents import LlmAgent, SequentialAgent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.adk.tools import google_search
from google.genai import types

## 2. Vertex AI Configuration

In [2]:
import os

os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"

PROJECT_ID = "qwiklabs-gcp-02-64fe8ee0c5bc"
LOCATION = "us-central1"

os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION

MODEL = "gemini-2.5-flash"

print("Vertex AI configuration complete.")
print(f"Project: {PROJECT_ID}")
print(f"Location: {LOCATION}")
print(f"Model: {MODEL}")

Vertex AI configuration complete.
Project: qwiklabs-gcp-02-64fe8ee0c5bc
Location: us-central1
Model: gemini-2.5-flash


## 3. Greeter Agent

The Greeter Agent receives the user's request and prepares it for the answer workflow.

In [3]:
greeter_agent = LlmAgent(
    name="greeter_agent",
    model=MODEL,
    description="Greets the user and restates the request for the workflow.",
    instruction="""
    You are the greeter for a travel planning workflow.

    Briefly acknowledge the user's request and restate what information
    the workflow should research.

    Keep the response concise.
    """,
    output_key="greeter_output",
)

### 3.1 Test the Greeter Agent

In [4]:
greeter_session_service = InMemorySessionService()

GREETER_APP_NAME = "greeter_test"
GREETER_USER_ID = "test_user"
GREETER_SESSION_ID = "greeter_session"

await greeter_session_service.create_session(
    app_name=GREETER_APP_NAME,
    user_id=GREETER_USER_ID,
    session_id=GREETER_SESSION_ID,
)

greeter_runner = Runner(
    agent=greeter_agent,
    app_name=GREETER_APP_NAME,
    session_service=greeter_session_service,
)

In [5]:
prompt = "I am planning a weekend trip to Denver. What are some good things to do?"

message = types.Content(
    role="user",
    parts=[types.Part(text=prompt)],
)

async for event in greeter_runner.run_async(
    user_id=GREETER_USER_ID,
    session_id=GREETER_SESSION_ID,
    new_message=message,
):
    print(f"EVENT AUTHOR: {event.author}")

    if event.is_final_response() and event.content:
        for part in event.content.parts:
            if getattr(part, "text", None):
                print(f"\nGREETER RESPONSE:\n{part.text}")

EVENT AUTHOR: greeter_agent

GREETER RESPONSE:
Okay, I can help you with that! The workflow will research good things to do for a weekend trip to Denver.


## 4. Search Agent

The Search Agent uses the ADK built-in Google Search tool to gather current information needed to answer the user's request.

In [6]:
search_agent = LlmAgent(
    name="search_agent",
    model=MODEL,
    description="Uses Google Search to research the user's travel question.",
    instruction="""
    You are the research agent in a travel planning workflow.

    Use Google Search to gather useful information that answers
    the user's request.

    Base your response on the search results.

    Provide a clear initial answer that can later be reviewed
    and improved by other agents in the workflow.
    """,
    tools=[google_search],
    output_key="search_output",
)

### 4.1 Test the Search Agent

In [7]:
search_session_service = InMemorySessionService()

SEARCH_APP_NAME = "search_test"
SEARCH_USER_ID = "test_user"
SEARCH_SESSION_ID = "search_session"

await search_session_service.create_session(
    app_name=SEARCH_APP_NAME,
    user_id=SEARCH_USER_ID,
    session_id=SEARCH_SESSION_ID,
)

search_runner = Runner(
    agent=search_agent,
    app_name=SEARCH_APP_NAME,
    session_service=search_session_service,
)

In [8]:
prompt = "I am planning a weekend trip to Denver. What are some good things to do?"

message = types.Content(
    role="user",
    parts=[types.Part(text=prompt)],
)

async for event in search_runner.run_async(
    user_id=SEARCH_USER_ID,
    session_id=SEARCH_SESSION_ID,
    new_message=message,
):
    print(f"EVENT AUTHOR: {event.author}")

    if event.is_final_response() and event.content:
        for part in event.content.parts:
            if getattr(part, "text", None):
                print(f"\nSEARCH RESPONSE:\n{part.text}")

EVENT AUTHOR: search_agent

SEARCH RESPONSE:
For a weekend trip to Denver, you'll find a blend of outdoor adventures, cultural experiences, and vibrant city life. Here are some highly recommended activities and attractions to consider:

**Iconic Attractions & Culture:**
*   **Red Rocks Park and Amphitheatre** is a must-visit for its stunning geological formations, hiking trails, and potential for catching a concert. It's considered the only naturally occurring, acoustically perfect amphitheater in the world, located just 15 miles west of Denver.
*   Explore **Meow Wolf Denver** for an immersive and unique art experience.
*   Wander through the diverse gardens at the **Denver Botanic Gardens**.
*   Visit the **Denver Art Museum**, known for its extensive collection, including Indigenous art and works by artists like Van Gogh and Winslow Homer.
*   Discover **Larimer Square**, Denver's oldest block, featuring Victorian buildings, boutiques, and diverse dining options.
*   Take a tour of 

## 5. Critique Agent

The Critique Agent reviews the Search Agent's initial response and identifies ways it can be improved before the final answer is produced.

In [9]:
critique_agent = LlmAgent(
    name="critique_agent",
    model=MODEL,
    description="Reviews the initial travel answer and recommends improvements.",
    instruction="""
    You are a critical reviewer in a travel planning workflow.

    Review the initial answer below:

    {search_output}

    Identify specific ways the answer could be improved.

    Consider:
    - relevance to the user's request
    - organization and clarity
    - whether the answer is too long or repetitive
    - whether the recommendations are practical for a weekend trip
    - whether important caveats or useful planning details are missing

    Do not rewrite the answer yet.
    Provide concise, actionable critique for the Refine Agent.
    """,
    output_key="critique_output",
)

### 5.1 Test the Search and Critique Workflow

Before building the complete workflow, test that the Search Agent's output is correctly passed to the Critique Agent.

In [10]:
search_critique_workflow = SequentialAgent(
    name="search_critique_workflow",
    description="Searches for an answer and then critiques the initial response.",
    sub_agents=[
        search_agent,
        critique_agent,
    ],
)

/tmp/ipykernel_78610/1584752507.py:1: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  search_critique_workflow = SequentialAgent(


In [11]:
workflow_session_service = InMemorySessionService()

WORKFLOW_APP_NAME = "search_critique_test"
WORKFLOW_USER_ID = "test_user"
WORKFLOW_SESSION_ID = "search_critique_session"

await workflow_session_service.create_session(
    app_name=WORKFLOW_APP_NAME,
    user_id=WORKFLOW_USER_ID,
    session_id=WORKFLOW_SESSION_ID,
)

workflow_runner = Runner(
    agent=search_critique_workflow,
    app_name=WORKFLOW_APP_NAME,
    session_service=workflow_session_service,
)

In [12]:
prompt = "I am planning a weekend trip to Denver. What are some good things to do?"

message = types.Content(
    role="user",
    parts=[types.Part(text=prompt)],
)

async for event in workflow_runner.run_async(
    user_id=WORKFLOW_USER_ID,
    session_id=WORKFLOW_SESSION_ID,
    new_message=message,
):
    print(f"EVENT AUTHOR: {event.author}")

    if event.content and event.content.parts:
        for part in event.content.parts:
            if getattr(part, "text", None):
                print(f"\n{event.author.upper()} OUTPUT:")
                print(part.text)

EVENT AUTHOR: search_agent

SEARCH_AGENT OUTPUT:
For a weekend trip to Denver, you'll find a wide array of activities ranging from historical exploration to outdoor adventures and cultural experiences.

Here are some recommended things to do:
*   **Explore Historical and Cultural Sites:** Visit the Colorado State Capitol, where you can take a tour. Other historical attractions include the History Colorado Center, Four Mile Historic Park, Colorado Railroad Museum, and the Buffalo Bill Museum & Grave. You can also consider touring the Denver Mint. For a deeper dive into local history, the Molly Brown House Museum offers guided tours.
*   **Enjoy Denver's Parks and Green Spaces:** Take a walk or go pedal boating in Washington Park, which features two lakes. City Park is another premier option, offering trails, lakes, and housing the Denver Zoo and the Museum of Nature & Science. Denver is also bike-friendly with many scenic trails, such as the Cherry Creek Trail.
*   **Visit Iconic Landma

## 6. Refine Agent

The Refine Agent rewrites the initial answer using the Critique Agent's suggestions to produce the final response.

In [13]:
refine_agent = LlmAgent(
    name="refine_agent",
    model=MODEL,
    description="Refines the initial travel answer using the critique.",
    instruction="""
    You are the final editor in a travel planning workflow.

    Initial answer:

    {search_output}

    Critique:

    {critique_output}

    Rewrite the initial answer using the critique.

    Requirements:
    - Keep the most useful recommendations.
    - Make the answer practical for a weekend trip.
    - Improve organization and clarity.
    - Remove unnecessary or repetitive information.
    - Include useful planning details when appropriate.
    - Do not mention the critique or the workflow in the final answer.
    - Return only the polished final response.
    """,
    output_key="refined_output",
)

## 7. Complete Sequential Workflow

The complete workflow uses a `SequentialAgent` to execute the four specialized agents in order:

**Greeter → Search → Critique → Refine**

Each agent performs one specific task, and intermediate results are stored in session state for use by later agents.

Create Fresh agent objects

In [15]:
greeter_agent_final = LlmAgent(
    name="greeter_agent",
    model=MODEL,
    description="Greets the user and restates the request for the workflow.",
    instruction="""
    You are the greeter for a travel planning workflow.

    Briefly acknowledge the user's request and restate what information
    the workflow should research.

    Keep the response concise.
    """,
    output_key="greeter_output",
)

search_agent_final = LlmAgent(
    name="search_agent",
    model=MODEL,
    description="Uses Google Search to research the user's travel question.",
    instruction="""
    You are the research agent in a travel planning workflow.

    Use Google Search to gather useful information that answers
    the user's request.

    Base your response on the search results.

    Provide a clear initial answer that can later be reviewed
    and improved by other agents in the workflow.
    """,
    tools=[google_search],
    output_key="search_output",
)

critique_agent_final = LlmAgent(
    name="critique_agent",
    model=MODEL,
    description="Reviews the initial travel answer and recommends improvements.",
    instruction="""
    You are a critical reviewer in a travel planning workflow.

    Review the initial answer below:

    {search_output}

    Identify specific ways the answer could be improved.

    Consider:
    - relevance to the user's request
    - organization and clarity
    - whether the answer is too long or repetitive
    - whether the recommendations are practical for a weekend trip
    - whether important caveats or useful planning details are missing

    Do not rewrite the answer yet.
    Provide concise, actionable critique for the Refine Agent.
    """,
    output_key="critique_output",
)

refine_agent_final = LlmAgent(
    name="refine_agent",
    model=MODEL,
    description="Refines the initial travel answer using the critique.",
    instruction="""
    You are the final editor in a travel planning workflow.

    Initial answer:

    {search_output}

    Critique:

    {critique_output}

    Rewrite the initial answer using the critique.

    Requirements:
    - Keep the most useful recommendations.
    - Make the answer practical for a weekend trip.
    - Improve organization and clarity.
    - Remove unnecessary or repetitive information.
    - Include useful planning details when appropriate.
    - Do not mention the critique or the workflow in the final answer.
    - Return only the polished final response.
    """,
    output_key="refined_output",
)

Build final workflow

In [16]:
answer_workflow = SequentialAgent(
    name="answer_workflow",
    description=(
        "Greets the user, researches the request, critiques the initial "
        "answer, and produces a refined final response."
    ),
    sub_agents=[
        greeter_agent_final,
        search_agent_final,
        critique_agent_final,
        refine_agent_final,
    ],
)

/tmp/ipykernel_78610/4212829256.py:1: DeprecationWarning: SequentialAgent is deprecated in favor of Workflow and will be removed in a future version. Workflow cannot yet be used as an LlmAgent sub-agent.
  answer_workflow = SequentialAgent(


### 7.2 Create the Full Workflow Test Runner

Create a dedicated session and runner for testing the complete
Greeter → Search → Critique → Refine workflow.

In [17]:
full_test_session_service = InMemorySessionService()

FULL_TEST_APP_NAME = "full_workflow_test"
FULL_TEST_USER_ID = "test_user"
FULL_TEST_SESSION_ID = "full_workflow_session"

await full_test_session_service.create_session(
    app_name=FULL_TEST_APP_NAME,
    user_id=FULL_TEST_USER_ID,
    session_id=FULL_TEST_SESSION_ID,
)

full_test_runner = Runner(
    agent=answer_workflow,
    app_name=FULL_TEST_APP_NAME,
    session_service=full_test_session_service,
)

7.3 Final Test

In [19]:
prompt = "I am planning a weekend trip to Denver. What are some good things to do?"

message = types.Content(
    role="user",
    parts=[types.Part(text=prompt)],
)

final_response = ""

print(f"USER: {prompt}")
print("=" * 70)

async for event in full_test_runner.run_async(
    user_id=FULL_TEST_USER_ID,
    session_id=FULL_TEST_SESSION_ID,
    new_message=message,
):
    print(f"\nEVENT AUTHOR: {event.author}")

    if event.content and event.content.parts:
        for part in event.content.parts:
            if getattr(part, "text", None):

                if event.author != "refine_agent":
                    print(f"\n{event.author.upper()} OUTPUT:")
                    print(part.text)

                if event.author == "refine_agent":
                    final_response = part.text

print("\n" + "=" * 70)
print("FINAL REFINED RESPONSE:")
print("=" * 70)
print(final_response)

USER: I am planning a weekend trip to Denver. What are some good things to do?

EVENT AUTHOR: greeter_agent

GREETER_AGENT OUTPUT:
Understood! I will research recommendations for activities and attractions for your weekend trip to Denver.

EVENT AUTHOR: search_agent

SEARCH_AGENT OUTPUT:
Denver offers a wide array of activities and attractions perfect for a weekend getaway, encompassing outdoor adventures, cultural experiences, and a vibrant food scene.

**Outdoor Activities:**
Known for its 300 days of sunshine, Denver provides numerous opportunities for outdoor enthusiasts. You can explore the city's extensive network of 850 miles of paved biking and walking trails. Consider walking the Mile High Trail in City Park, a 3.1-mile route that follows the city's 5280 contour line, placing you exactly a mile high. Kayaking is another popular option, with various rivers and lakes like the South Platte River available for tranquil paddling experiences, and rentals often available through serv